# ITT Pulmon de Oriente 2026 — 5 Dimensiones


## Descripcion General


**Zona:** Pulmon de Oriente — Comunas 13, 14, 15 y 16, Cali  
**Periodo:** 2023–2026 T1  
**Metodologia:** `ref_min/ref_max` fijos | Paleta Okabe-Ito | Heatmaps cividis  
**Formula ITT (5 dims):**

```
ITT = 0.27 x Seguridad + 0.22 x Movilidad + 0.19 x Desarrollo Social + 0.17 x Entorno Urbano + 0.15 x Act. Economica
```

| Nivel | Rango | Descripcion |
|---|---|---|
| Activacion | 0–40 | Condiciones criticas |
| Consolidacion | 40–60 | Avances parciales |
| Transformacion | 60–80 | Mejora consolidada |
| Escala | 80–100 | Transformacion robusta |


In [ ]:
# Descomentar en Colab
# !pip install geopandas pyproj shapely openpyxl matplotlib seaborn folium rasterio pillow gdown -q
import subprocess, sys
def check_pkg(pkg):
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
for p in ['geopandas','pyproj','shapely','openpyxl','seaborn','folium','rasterio','Pillow','gdown']:
    check_pkg(p)
print('Dependencias verificadas')


In [ ]:
import json, os, re, warnings
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import folium
warnings.filterwarnings('ignore')

# ── Paleta Okabe-Ito (accesible para daltonicos) ─────────────────────
OKI_AZUL     = '#0072B2'
OKI_NARANJA  = '#E69F00'
OKI_VERDE    = '#009E73'
OKI_AMARILLO = '#F0E442'
OKI_AZUL_C   = '#56B4E9'
OKI_BERMELL  = '#D55E00'
OKI_MORADO   = '#CC79A7'
OKI_GRIS     = '#333333'

C_SEG = OKI_AZUL; C_MOV = OKI_NARANJA; C_COH = OKI_VERDE
C_ITT = OKI_GRIS; BG = '#F4F6F9'

NIVEL_COLORS = {
    'Activacion':     OKI_BERMELL,
    'Consolidacion':  OKI_NARANJA,
    'Transformacion': OKI_VERDE,
    'Escala':         OKI_AZUL,
}

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})
print('Configuracion visual lista - Paleta Okabe-Ito activa')


In [ ]:
import os, zipfile, subprocess, glob
from pathlib import Path

# ── Clonar / actualizar repo principal ────────────────────────────────
if os.path.exists('/content/itt_repos_cali'):
    !git -C /content/itt_repos_cali pull
else:
    !git clone https://github.com/Pabandres85/itt_repos_cali.git /content/itt_repos_cali

REPO       = Path('/content/itt_repos_cali')
PULMON_DIR = REPO / 'data' / 'itt_pulmon_oriente'
ZIP_PATH   = PULMON_DIR / 'Pulmon_De_Oriente_2026.zip'

# ── Extraer ZIP ───────────────────────────────────────────────────────
if ZIP_PATH.exists():
    # Siempre extraer para garantizar estructura completa
    print(f'Extrayendo {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1024:.1f} KB)...')
    with zipfile.ZipFile(str(ZIP_PATH), 'r') as zf:
        zf.extractall(str(PULMON_DIR))
    print('Extraccion completada')
else:
    print(f'AVISO: ZIP no encontrado en {ZIP_PATH}')

# ── Buscar dinamicamente la carpeta de GeoJSON ────────────────────────
# El ZIP puede extraer con nombres ligeramente diferentes segun encoding
print(f'\nContenido de {PULMON_DIR}:')
for p in sorted(PULMON_DIR.iterdir()):
    print(f'  {"[DIR]" if p.is_dir() else "[FILE]"} {p.name}')

# Buscar carpeta que contenga 'Pulmon' y sea directorio
DATA_DIR = None
for p in PULMON_DIR.iterdir():
    if p.is_dir() and 'Pulmon' in p.name:
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError(f'No se encontro carpeta Pulmon* en {PULMON_DIR}')

print(f'\nDATA_DIR encontrado: {DATA_DIR.name}')
print(f'Contenido de DATA_DIR:')
for p in sorted(DATA_DIR.iterdir()):
    print(f'  {"[DIR]" if p.is_dir() else "[FILE]"} {p.name}')
    if p.is_dir():
        for pp in sorted(p.iterdir()):
            print(f'    {"[DIR]" if pp.is_dir() else "[FILE]"} {pp.name}')

# Buscar carpeta de GeoJSON — puede estar en DATA_DIR o un nivel mas adentro
GJ_DIR = None
# Nivel 1: directamente en DATA_DIR
for p in DATA_DIR.iterdir():
    if p.is_dir() and 'geojson' in p.name.lower():
        GJ_DIR = p
        break

# Nivel 2: buscar en subcarpetas de DATA_DIR (doble anidacion del ZIP)
if GJ_DIR is None:
    for p in DATA_DIR.iterdir():
        if p.is_dir():
            for pp in p.iterdir():
                if pp.is_dir() and 'geojson' in pp.name.lower():
                    GJ_DIR = pp
                    break
            if GJ_DIR: break

# Nivel alternativo: si las carpetas de dimension estan directamente en DATA_DIR
if GJ_DIR is None:
    # Verificar si DATA_DIR contiene directamente carpetas como 1_Dimension_Seguridad
    has_dims = any('dimension' in p.name.lower() or 'seguridad' in p.name.lower()
                   for p in DATA_DIR.iterdir() if p.is_dir())
    if has_dims:
        GJ_DIR = DATA_DIR  # Las dimensiones estan directamente aqui
        print('\nNota: Carpetas de dimension encontradas directamente en DATA_DIR (sin Geojson_* intermedio)')

if GJ_DIR is None:
    raise FileNotFoundError(
        f'No se encontro carpeta con GeoJSON en ninguno de los niveles esperados.\n'
        f'Revisa el output de arriba para ver la estructura real.'
    )

print(f'\nGJ_DIR encontrado: {GJ_DIR.name}')
print('Carpetas de dimensiones:')
for p in sorted(GJ_DIR.iterdir()):
    print(f'  {p.name}')

# Para ejecucion local descomentar:
# REPO       = Path('.')
# PULMON_DIR = REPO / 'data' / 'itt_pulmon_oriente'
# DATA_DIR   = PULMON_DIR / 'Pulmon_De_Oriente_2026'
# GJ_DIR     = DATA_DIR / 'Geojson_Pulmon_De_Oriente_2026'


In [ ]:
from pathlib import Path
import os

# ── Buscar subcarpetas por patron (evita problemas de encoding) ───────
def find_subdir(parent, pattern):
    """Busca una subcarpeta cuyo nombre contenga el patron dado."""
    for p in parent.iterdir():
        if p.is_dir() and pattern.lower() in p.name.lower():
            return p
    return None

D = GJ_DIR
dir_seg = find_subdir(D, '1_Dimension_Seguridad') or find_subdir(D, 'Seguridad')
dir_coh = find_subdir(D, 'Cohesion_Social') or find_subdir(D, 'Cohesion')
dir_mov = find_subdir(D, 'Dimension_Movilidad') or find_subdir(D, 'Movilidad')
dir_eco = find_subdir(D, 'Desarrollo_Economico') or find_subdir(D, 'Economico')
dir_ent = find_subdir(D, 'Entorno_Urbano') or find_subdir(D, 'Entorno')
dir_edu = find_subdir(D, 'Educacion')

# Buscar archivos GeoJSON por patron dentro de cada subcarpeta
def find_gj(folder, pattern):
    """Busca el primer archivo .geojson cuyo nombre contenga el patron."""
    if folder is None: return None
    for f in folder.iterdir():
        if f.suffix.lower() == '.geojson' and pattern.lower() in f.name.lower():
            return str(f)
    return None

# Buscar shapefile del poligono
shape_dir = find_subdir(DATA_DIR, 'shape')
poligono_path = None
if shape_dir:
    for f in shape_dir.iterdir():
        if 'poligono' in f.name.lower() and f.suffix == '.shp':
            poligono_path = str(f)
            break

PATHS = {
    'poligono':    poligono_path,
    'homicidios':  find_gj(dir_seg, 'homicidio'),
    'hurtos':      find_gj(dir_seg, 'hurto'),
    'comparendos': find_gj(dir_seg, 'comparendo'),
    'vif':         find_gj(dir_coh, 'violencia_intrafamiliar'),
    'siniestros':  find_gj(dir_mov, 'SINIESTROS'),
    'arboles':     find_gj(dir_ent, 'ARBOREO'),
    'registro_mercantil': find_gj(dir_eco, 'registro_mercantil'),
}

ANIOS       = [2023, 2024, 2025, 2026]
ANO_CORTE   = 2026
ZONA_NOMBRE = 'Pulmon de Oriente \u2014 Comunas 13, 14, 15, 16'

# ── Pesos ITT 5 dimensiones ──────────────────────────────────────────
PESOS = {
    'Seguridad': 0.27,
    'Movilidad': 0.22,
    'DesSocial': 0.19,
    'EntornoU':  0.17,
    'DesEco':    0.15,
}

# ── Umbrales calibrados para zona grande (Pulmon de Oriente >100K hab) ─
REFS = {
    'homicidios':     (5,   50,  True,  'Homicidios trimestrales en el poligono'),
    'hurtos':         (200, 450, True,  'Hurtos trimestrales en el poligono'),
    'siniestralidad': (28,  75,  True,  'Siniestros viales trimestrales'),
    'lesionados':     (20,  65,  True,  'Accidentes con lesionados trimestrales'),
    'mortales':       (1,   10,  True,  'Accidentes mortales trimestrales'),
    'vif':            (60,  200, True,  'Violencia intrafamiliar trimestral'),
    'rinas':          (20,  160, True,  'Rinas / conflictividad trimestral'),
    'spa':            (0,   300, True,  'Sustancias psicoactivas (comparendos) trimestrales'),
}

# ── Referentes provisionales (fallback) ──────────────────────────────
REF_ENTORNO_U      = 39.2   # se sobreescribe si NDVI real disponible
REF_EDUC_DES       = 54.9   # se sobreescribe si SIMAT/SIMPADE disponible
REF_DES_ECO        = 50.0   # se sobreescribe con Registro Mercantil
REF_VULNERABILIDAD = 54.1   # contexto, no entra al score

print(f'Zona: {ZONA_NOMBRE}  |  Periodo: {ANIOS[0]}-{ANIOS[-1]}')
for nombre, path in PATHS.items():
    mark = '\u2705' if os.path.exists(path) else '\u274c'
    print(f'  {mark}  {nombre}: {os.path.basename(path)}')
print()
print('Pesos ITT (5 dims):', ' | '.join(f'{k}={v:.0%}' for k, v in PESOS.items()))
print(f'Suma pesos: {sum(PESOS.values()):.2f}')
print()
print('Umbrales ref_min/ref_max:')
for ind, (rmin, rmax, inv, desc) in REFS.items():
    print(f'  {ind:18s}  [{rmin:>3} - {rmax:>4}]  inv={inv}  \u2014 {desc}')


In [ ]:
def load_gj(path):
    """Carga un GeoJSON como diccionario Python."""
    with open(path, encoding='utf-8') as f:
        return json.load(f)

# ── Cargar poligono ───────────────────────────────────────────────────
gdf_pulmon     = gpd.read_file(PATHS['poligono'])
gdf_pulmon_wgs = gdf_pulmon.to_crs('EPSG:4326')
area_ha = gdf_pulmon.to_crs('EPSG:3115').geometry.area.sum() / 10_000
print(f'Poligono cargado: {len(gdf_pulmon)} feature(s) | Area total: {area_ha:.1f} ha')

# ── Cargar capas de eventos ───────────────────────────────────────────
raw_hom  = load_gj(PATHS['homicidios'])
raw_hur  = load_gj(PATHS['hurtos'])
raw_sin  = load_gj(PATHS['siniestros'])
raw_vif  = load_gj(PATHS['vif'])
raw_comp = load_gj(PATHS['comparendos'])
raw_arb  = load_gj(PATHS['arboles'])
raw_rm   = load_gj(PATHS['registro_mercantil'])

print()
print('Registros cargados:')
for n, r in [('Homicidios',raw_hom),('Hurtos',raw_hur),('Siniestros',raw_sin),
             ('VIF',raw_vif),('Comparendos',raw_comp),('Arboles',raw_arb),
             ('Reg. Mercantil',raw_rm)]:
    print(f'  {n:18s}: {len(r["features"]):>5} registros')


In [ ]:
def procesar(raw, col_fecha, filtro=None, filtro_col=None, formato_fecha=None, startswith=False):
    """Extrae propiedades, deduplica, filtra y parsea fechas."""
    df = pd.DataFrame([f['properties'] for f in raw['features']])
    _ng = [col for col in df.columns if col not in ('geom','geometry','the_geom')]
    _b  = len(df)
    df  = df.drop_duplicates(subset=_ng).reset_index(drop=True)
    if _b != len(df):
        print(f'  [dedup] {_b-len(df)} duplicados removidos ({_b}->{len(df)})')

    if filtro and filtro_col:
        if startswith:
            df = df[df[filtro_col].astype(str).str.startswith(filtro)].copy()
        else:
            df = df[df[filtro_col] == filtro].copy()

    df['_fecha'] = pd.to_datetime(df[col_fecha], format=formato_fecha, errors='coerce')
    df['a\u00f1o']      = df['_fecha'].dt.year
    df['trimestre'] = df['_fecha'].dt.quarter
    return df[df['a\u00f1o'].isin(ANIOS)].copy()

def agg_anual(df):
    return df.groupby('a\u00f1o').size().reindex(ANIOS, fill_value=0)

def agg_trim(df):
    idx = pd.MultiIndex.from_product([ANIOS,[1,2,3,4]], names=['a\u00f1o','trimestre'])
    return df.groupby(['a\u00f1o','trimestre']).size().reindex(idx, fill_value=0)

def score_ref(valor, ref_min, ref_max, inverso):
    """Normaliza un valor con umbrales fijos ref_min/ref_max (0-100)."""
    if ref_max == ref_min:
        return 100.0
    raw = np.clip((valor - ref_min) / (ref_max - ref_min) * 100, 0, 100)
    return 100 - raw if inverso else raw

def safe_pct(new, old):
    if old == 0: return 0.0
    return (new - old) / old * 100

def arrow(pct, inv=True):
    if abs(pct) < 1: return 'Sin cambio', 'gray'
    if inv:
        return (f'\u2193 {abs(pct):.1f}%','#2E7D32') if pct<0 else (f'\u2191 {abs(pct):.1f}%','#C62828')
    else:
        return (f'\u2191 {abs(pct):.1f}%','#2E7D32') if pct>0 else (f'\u2193 {abs(pct):.1f}%','#C62828')

def clasificar(v):
    if pd.isna(v): return 'S/D'
    if v < 40:   return 'Activacion'
    elif v < 60: return 'Consolidacion'
    elif v < 80: return 'Transformacion'
    else:        return 'Escala'

print('Funciones utilitarias definidas')


In [ ]:
# ── Procesar cada indicador ───────────────────────────────────────────
df_hom = procesar(raw_hom, 'fechah', formato_fecha='%m/%d/%Y')
df_hur = procesar(raw_hur, 'fecha_hech')
df_sin = procesar(raw_sin, 'Fecha')
df_les = df_sin[df_sin['Tipo_Confi'] == 'Lesiones'].copy()
df_mor = df_sin[df_sin['Tipo_Confi'] == 'Mortal'].copy()
df_vif = procesar(raw_vif, 'fecha_hech')
df_rin = procesar(raw_comp, 'fecha_hech', filtro='RI', filtro_col='agrupado', startswith=True)

# ── Serie anual base ──────────────────────────────────────────────────
base = pd.DataFrame({'a\u00f1o': ANIOS})
for nombre, df_src in [('homicidios',df_hom),('hurtos',df_hur),
                       ('siniestralidad',df_sin),('lesionados',df_les),
                       ('mortales',df_mor),('vif',df_vif),('rinas',df_rin)]:
    base[nombre] = agg_anual(df_src).values

# ── Serie trimestral ──────────────────────────────────────────────────
idx_t = pd.MultiIndex.from_product([ANIOS,[1,2,3,4]], names=['a\u00f1o','trimestre'])
corr_trim = pd.DataFrame(index=idx_t).reset_index()
for nombre, df_src in [('homicidios',df_hom),('hurtos',df_hur),
                       ('siniestralidad',df_sin),('lesionados',df_les),
                       ('mortales',df_mor),('vif',df_vif),('rinas',df_rin)]:
    ser = agg_trim(df_src).reset_index()
    ser.columns = ['a\u00f1o','trimestre',nombre]
    corr_trim = corr_trim.merge(ser, on=['a\u00f1o','trimestre'], how='left').fillna({nombre:0})
corr_trim['periodo'] = corr_trim['a\u00f1o'].astype(str) + '-Q' + corr_trim['trimestre'].astype(str)

# ── Mascara 2026: solo Q1 disponible para DATIC ──────────────────────
mask_q2plus = (corr_trim['a\u00f1o'] == 2026) & (corr_trim['trimestre'] > 1)
corr_trim.loc[mask_q2plus, ['homicidios','hurtos','vif','rinas']] = np.nan

# Siniestros 2026: detectar trimestres disponibles
_q_sin_2026 = int(df_sin[df_sin['a\u00f1o']==2026]['trimestre'].nunique()) if len(df_sin[df_sin['a\u00f1o']==2026]) > 0 else 0
mask_sin_fut = (corr_trim['a\u00f1o'] == 2026) & (corr_trim['trimestre'] > max(_q_sin_2026, 1))
corr_trim.loc[mask_sin_fut, ['siniestralidad','lesionados','mortales']] = np.nan

print('Serie base construida:')
print(base.to_string(index=False))
print(f'\nTrimestres con datos en 2026 (DATIC): Q1 | Siniestros: Q1-Q{_q_sin_2026}')


In [ ]:
# ── Normalizacion con refs fijos ──────────────────────────────────────
for ind, (rmin, rmax, inv, desc) in REFS.items():
    if ind in base.columns:
        base[f'score_{ind}'] = base[ind].apply(lambda v: score_ref(v, rmin, rmax, inv))

# ── SPA (Sustancias Psicoactivas) desde comparendos ──────────────────
_df_spa = pd.DataFrame([f['properties'] for f in raw_comp['features']])
_df_spa = _df_spa[_df_spa['agrupado'] == 'SUSTANCIAS PSICOACTIVAS'].copy()
_df_spa['_fecha'] = pd.to_datetime(_df_spa['fecha_hech'], errors='coerce')
_df_spa['a\u00f1o']    = _df_spa['_fecha'].dt.year
_spa_anual = _df_spa.groupby('a\u00f1o').size().reindex(ANIOS, fill_value=0)

# Anualizar SPA 2026 si solo hay Q1
_q_spa_2026 = int(_df_spa[_df_spa['a\u00f1o']==2026]['_fecha'].dt.quarter.nunique()) if len(_df_spa[_df_spa['a\u00f1o']==2026])>0 else 0
if 0 < _q_spa_2026 < 4:
    _spa_anual[2026] = round(_spa_anual[2026] * 4 / _q_spa_2026)
    print(f'  SPA 2026: {_q_spa_2026}T disponibles -> anualizado x{4/_q_spa_2026:.1f}')
base['spa']       = _spa_anual.values
base['score_spa'] = base['spa'].apply(lambda v: score_ref(v, 0, 300, True))

# ── Scores por dimension ──────────────────────────────────────────────
base['score_seguridad'] = (base['score_homicidios'] + base['score_hurtos']) / 2
base['score_movilidad'] = (base['score_siniestralidad'] + base['score_lesionados'] + base['score_mortales']) / 3
base['score_cohesion']  = (base['score_vif'] + base['score_rinas'] + base['score_spa']) / 3
base['score_entorno_u'] = REF_ENTORNO_U    # provisional, se sobreescribe mas adelante
base['score_educ_des']  = REF_EDUC_DES     # provisional, se sobreescribe mas adelante
base['score_des_social'] = (base['score_cohesion'] + base['score_educ_des']) / 2
base['score_des_eco']   = REF_DES_ECO      # provisional, se sobreescribe mas adelante

# ── Calculo ITT con redistribucion de peso si faltan dims ─────────────
def _itt_row(row):
    dims = {
        'Seguridad': row.get('score_seguridad', np.nan),
        'Movilidad': row.get('score_movilidad', np.nan),
        'DesSocial': row.get('score_des_social', np.nan),
        'EntornoU':  row.get('score_entorno_u', np.nan),
        'DesEco':    row.get('score_des_eco', np.nan),
    }
    total_peso = sum(PESOS[k] for k, v in dims.items() if not pd.isna(v))
    if total_peso == 0: return np.nan
    return sum(PESOS[k] * v for k, v in dims.items() if not pd.isna(v)) / total_peso

base['ITT']   = base.apply(_itt_row, axis=1)
base['nivel'] = base['ITT'].apply(clasificar)

print('\nITT Pulmon de Oriente \u2014 Normalizacion provisional | 5 dimensiones')
cols_show = ['a\u00f1o','score_seguridad','score_movilidad','score_des_social',
             'score_entorno_u','score_des_eco','ITT','nivel']
print(base[cols_show].round(1).to_string(index=False))


## Dimension Seguridad — Pulmon de Oriente

**Peso:** 27%  
**Indicadores:** Homicidios (inv) + Hurtos (inv)  
**Fuente:** DATIC 2023–2026 T1  
**Refs:** Homicidios [5–50] | Hurtos [200–450]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=BG)
fig.suptitle('Dimension Seguridad \u2014 Heatmap Trimestral | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
for ax, col, titulo in [(axes[0],'homicidios','Homicidios'),(axes[1],'hurtos','Hurtos')]:
    pivot = corr_trim.pivot(index='a\u00f1o', columns='trimestre', values=col)
    pivot.columns = ['Q1','Q2','Q3','Q4']
    mask_nan = pivot.isna()
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='cividis',
        linewidths=0.5, linecolor='#DEE2E6', ax=ax, annot_kws={'size':11},
        cbar_kws={'label':'Casos','shrink':0.8}, mask=mask_nan)
    ax.set_title(titulo, fontweight='bold', pad=8)
    ax.set_ylabel(''); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('itt_pulmon_heatmap_seg.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
periodos = corr_trim['periodo'].values
x = list(range(len(periodos)))

fig, axes = plt.subplots(1, 2, figsize=(16, 4), facecolor=BG)
fig.suptitle('Seguridad \u2014 Evolucion Trimestral | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
for ax, col, titulo, color in [(axes[0],'homicidios','Homicidios',OKI_BERMELL),
                                (axes[1],'hurtos','Hurtos',OKI_AZUL)]:
    vals = corr_trim[col].values.astype(float)
    ax.fill_between(x, vals, alpha=0.2, color=color)
    ax.plot(x, vals, color=color, linewidth=2.5, marker='o', markersize=4)
    for i, v in enumerate(vals):
        if pd.notna(v) and v > 0:
            ax.annotate(str(int(v)), (i, v), textcoords='offset points',
                        xytext=(0,7), ha='center', fontsize=7.5, fontweight='bold', color=color)
    ax.set_xticks(x[::2])
    ax.set_xticklabels(periodos[::2], rotation=45, ha='right', fontsize=8)
    ax.set_title(titulo, fontweight='bold', pad=8)
    ax.set_ylabel('Casos')
plt.tight_layout()
plt.savefig('itt_pulmon_linea_seg.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## Dimension Movilidad — Pulmon de Oriente

**Peso:** 22%  
**Indicadores:** Siniestralidad vial (inv) + Lesionados (inv) + Mortales (inv)  
**Fuente:** BD Siniestros 2023–2026  
**Refs:** Siniestralidad [28–75] | Lesionados [20–65] | Mortales [1–10]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4.6), facecolor=BG)
fig.suptitle('Dimension Movilidad \u2014 Heatmap Trimestral | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
for ax, col, titulo in [(axes[0],'siniestralidad','Siniestros Totales'),
                         (axes[1],'lesionados','Con Lesionados'),
                         (axes[2],'mortales','Mortales')]:
    pivot = corr_trim.pivot(index='a\u00f1o', columns='trimestre', values=col)
    pivot.columns = ['Q1','Q2','Q3','Q4']
    mask_nan = pivot.isna()
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='cividis',
        linewidths=0.5, linecolor='#DEE2E6', ax=ax, annot_kws={'size':11},
        cbar_kws={'label':'Casos','shrink':0.8}, mask=mask_nan)
    ax.set_title(titulo, fontweight='bold', pad=8)
    ax.set_ylabel(''); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('itt_pulmon_heatmap_mov.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4), facecolor=BG)
fig.suptitle('Movilidad \u2014 Evolucion Trimestral | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
for ax, col, titulo, color in [(axes[0],'siniestralidad','Siniestros',OKI_NARANJA),
                                (axes[1],'lesionados','Lesionados',OKI_BERMELL),
                                (axes[2],'mortales','Mortales',OKI_MORADO)]:
    vals = corr_trim[col].values.astype(float)
    ax.fill_between(x, vals, alpha=0.2, color=color)
    ax.plot(x, vals, color=color, linewidth=2.5, marker='o', markersize=4)
    for i, v in enumerate(vals):
        if pd.notna(v) and v > 0:
            ax.annotate(str(int(v)), (i, v), textcoords='offset points',
                        xytext=(0,7), ha='center', fontsize=7.5, fontweight='bold', color=color)
    ax.set_xticks(x[::2])
    ax.set_xticklabels(periodos[::2], rotation=45, ha='right', fontsize=8)
    ax.set_title(titulo, fontweight='bold', pad=8)
    ax.set_ylabel('Casos')
plt.tight_layout()
plt.savefig('itt_pulmon_linea_mov.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## Dimension Desarrollo Social — Pulmon de Oriente

**Peso:** 19%  
**Composicion:** Promedio(Cohesion Social, Educacion y Desarrollo)

### Cohesion Social
- VIF (inv) | Rinas (inv) | SPA (inv)
- Refs: VIF [60–200] | Rinas [20–160] | SPA [0–300]
- Fuente: DATIC 2023–2026 T1

### Educacion y Desarrollo
- Desercion (inv) | Repitencia (inv)
- Refs: Desercion [0%–15%] | Repitencia [0%–20%]
- Fuente: SIMAT / SIMPADE (descarga Drive)
- Fallback: REF_EDUC_DES = 54.9


In [ ]:
# ── Filtrar SPA desde comparendos ya cargados ─────────────────────────
df_comp_all = pd.DataFrame([f['properties'] for f in raw_comp['features']])
df_spa = df_comp_all[df_comp_all['agrupado'] == 'SUSTANCIAS PSICOACTIVAS'].copy()
df_spa['_fecha']    = pd.to_datetime(df_spa['fecha_hech'], errors='coerce')
df_spa['a\u00f1o']       = df_spa['_fecha'].dt.year
df_spa['trimestre'] = df_spa['_fecha'].dt.quarter

pivot_spa = (
    df_spa.groupby(['a\u00f1o','trimestre']).size()
    .unstack(fill_value=0)
    .reindex(index=ANIOS, fill_value=0)
    .reindex(columns=[1,2,3,4], fill_value=0)
    .astype(float)
)
pivot_spa.columns = ['Q1','Q2','Q3','Q4']
_q_max_spa = int(df_spa[df_spa['a\u00f1o']==2026]['trimestre'].max()) if len(df_spa[df_spa['a\u00f1o']==2026]) > 0 else 0
for q in range(_q_max_spa+1, 5):
    pivot_spa.loc[2026, f'Q{q}'] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(16, 4), facecolor=BG)
fig.suptitle('Sustancias Psicoactivas \u2014 Comparendos | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
mask_nan = pivot_spa.isna()
sns.heatmap(pivot_spa, annot=True, fmt='.0f', cmap='cividis',
    linewidths=0.5, linecolor='#DEE2E6', ax=axes[0],
    annot_kws={'size': 12, 'weight': 'bold'},
    cbar_kws={'label': 'Comparendos SPA', 'shrink': 0.8}, mask=mask_nan)
axes[0].set_title('Heatmap trimestral', fontweight='bold', pad=8)

periodos_spa = [f'{a}-Q{t}' for a in ANIOS for t in [1,2,3,4]]
vals_spa = [pivot_spa.loc[a, f'Q{t}'] if not pd.isna(pivot_spa.loc[a, f'Q{t}']) else np.nan
            for a in ANIOS for t in [1,2,3,4]]
x_spa = list(range(len(periodos_spa)))
axes[1].fill_between(x_spa, vals_spa, alpha=0.25, color=OKI_MORADO)
axes[1].plot(x_spa, vals_spa, color=OKI_MORADO, linewidth=2.5, marker='o', markersize=5)
axes[1].set_xticks(x_spa[::2])
axes[1].set_xticklabels(periodos_spa[::2], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Evolucion trimestral SPA', fontweight='bold', pad=8)
plt.tight_layout()
plt.savefig('itt_pulmon_spa.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Total SPA: {len(df_spa)} eventos')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=BG)
fig.suptitle('Cohesion Social \u2014 VIF y Rinas | Pulmon de Oriente',
             fontsize=13, fontweight='bold', color='#1B2631')
for ax, col, titulo in [(axes[0],'vif','Violencia Intrafamiliar'),(axes[1],'rinas','Rinas')]:
    pivot = corr_trim.pivot(index='a\u00f1o', columns='trimestre', values=col)
    pivot.columns = ['Q1','Q2','Q3','Q4']
    mask_nan = pivot.isna()
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='cividis',
        linewidths=0.5, linecolor='#DEE2E6', ax=ax, annot_kws={'size':11},
        cbar_kws={'label':'Casos','shrink':0.8}, mask=mask_nan)
    ax.set_title(titulo, fontweight='bold', pad=8)
    ax.set_ylabel(''); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('itt_pulmon_heatmap_cohesion.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── Vulnerabilidad activa: Sub PyE 2025 ──────────────────────────────
# Proxy: se filtra por comunas 13, 14, 15, 16 del poligono Pulmon de Oriente
PATH_SUBPYE = str(REPO / 'data' / 'referencia' / 'bienestar' /
                  'Caracterizaci\u00f3n Personas Sub PyE (2025).xlsx')

# Poblacion estimada Pulmon de Oriente (Comunas 13-16, ~450K hab segun DANE)
POB_PULMON   = 450_000
VULN_REF_MIN = 0
VULN_REF_MAX = 100

if os.path.exists(PATH_SUBPYE):
    df_sub = pd.read_excel(PATH_SUBPYE)
    COL_COMUNA = [c for c in df_sub.columns if 'comuna' in c.lower()]
    if COL_COMUNA:
        col_com = COL_COMUNA[0]
        # Filtrar comunas 13, 14, 15, 16
        mask_pulmon = df_sub[col_com].astype(str).str.extract(r'(\d+)')[0].astype(float).isin([13,14,15,16])
        df_pulmon = df_sub[mask_pulmon].copy()
    else:
        # Fallback: usar todo el dataset
        df_pulmon = df_sub.copy()
        print('  AVISO: No se encontro columna comuna, usando dataset completo')

    n_personas = len(df_pulmon)
    tasa_vuln  = n_personas / POB_PULMON * 1000
    raw_v      = np.clip((tasa_vuln - VULN_REF_MIN) / (VULN_REF_MAX - VULN_REF_MIN) * 100, 0, 100)
    score_vuln = round(100 - raw_v, 1)
    REF_VULNERABILIDAD = score_vuln
    print(f'Vulnerabilidad activa | Pulmon de Oriente (Sub PyE 2025)')
    print(f'  Personas: {n_personas}  |  Tasa: {tasa_vuln:.1f}/1.000 hab  |  Score: {score_vuln}')
else:
    print(f'  Archivo Sub PyE no encontrado: {PATH_SUBPYE}')
    print(f'  Usando REF_VULNERABILIDAD = {REF_VULNERABILIDAD} (provisorio)')


In [ ]:
# ── Educacion y Desarrollo ───────────────────────────────────────────
# Para Pulmon de Oriente se requiere identificar las sedes educativas
# dentro del poligono y obtener SIMAT/SIMPADE.
# La carpeta 6_Educacion/ esta vacia en los datos actuales.
# Se usa REF_EDUC_DES = 54.9 como fallback hasta tener datos SIMAT.

# Placeholder para cuando se disponga de datos:
# SIMAT_DRIVE_IDS = {2023: '...', 2024: '...', 2025: '...'}
# import gdown
# for anio, fid in SIMAT_DRIVE_IDS.items():
#     dest = Path(f'/content/simat_6a_dic_{anio}.xlsx')
#     gdown.download(id=fid, output=str(dest), quiet=False)

# DANE sedes en el poligono: pendiente de identificar

print('Educacion y Desarrollo:')
print(f'  Score provisional (fallback): {REF_EDUC_DES}')
print(f'  Fuente: ITT Pulmon de Oriente T4-2025')
print(f'  Estado: pendiente de datos SIMAT/SIMPADE para sedes en el poligono')


## Dimension Entorno Urbano — Pulmon de Oriente

**Peso:** 17%  
**Indicadores:** % Area verde NDVI (pos) + Densidad de arboles (pos)  
**Fuentes:**
- Rasters NDVI Sentinel-2 (pendiente de inclusion en repo)
- Censo Arboreo DAGMA (GeoJSON disponible)

**Refs:** % Verde [0–30%] | Arboles [0–50 arb/ha]

**Nota:** Los rasters NDVI para Pulmon de Oriente no estan en el repo aun.  
Se usa el score provisional 39.2 hasta disponer de imagenes Sentinel-2 procesadas.  
Cuando se incluyan TIF anuales, el calculo sera identico a Roosevelt/Barrio Obrero.


In [ ]:
# ── Entorno Urbano: Arboles DAGMA ─────────────────────────────────────
REF_VERDE_MIN,   REF_VERDE_MAX   = 0, 30
REF_ARBOLES_MIN, REF_ARBOLES_MAX = 0, 50

gdf_arb = gpd.GeoDataFrame.from_features(raw_arb['features'], crs='EPSG:4326')
n_arboles = len(gdf_arb)
dens_arboles_ha = n_arboles / area_ha if area_ha > 0 else 0
score_arboles_fijo = round(score_ref(dens_arboles_ha, REF_ARBOLES_MIN, REF_ARBOLES_MAX, False), 1)

print(f'Censo Arboreo DAGMA:')
print(f'  Arboles en poligono: {n_arboles}')
print(f'  Area poligono: {area_ha:.1f} ha')
print(f'  Densidad: {dens_arboles_ha:.2f} arb/ha')
print(f'  Score arboles: {score_arboles_fijo}  (ref 0-50 arb/ha)')

# ── NDVI: placeholder para rasters ────────────────────────────────────
# Cuando se incluyan TIF en data/itt_pulmon_oriente/.../5_Entorno_Urbano/NDVI/
# import rasterio
# ndvi_stats = {}
# for anio in ANIOS:
#     tif_path = GJ_DIR / '5_Entorno_Urbano' / 'NDVI' / f'Pulmon_Oriente_ndvi_{anio}.tif'
#     if tif_path.exists():
#         with rasterio.open(str(tif_path)) as src:
#             data = src.read(1)
#             valid = data[data != src.nodata] if src.nodata else data.flatten()
#             pct_verde = (valid >= 0.20).sum() / len(valid) * 100
#             ndvi_stats[anio] = {'ndvi_mean': valid.mean(), 'pct_verde': pct_verde}

# Por ahora: score entorno con solo arboles + referente NDVI provisional
rows_eu = []
for anio in ANIOS:
    # Sin NDVI: usar solo arboles como indicador real + ref provisional para verde
    sc_eu = score_arboles_fijo  # solo arboles por ahora
    rows_eu.append({'a\u00f1o': anio, 'score_arboles': score_arboles_fijo,
                    'score_verde': np.nan, 'score_entorno_u': sc_eu})

df_eu = pd.DataFrame(rows_eu)
print()
print('Entorno Urbano (provisional \u2014 solo arboles):')
print(df_eu.to_string(index=False))

# Actualizar base con score real de arboles
for _, row in df_eu.iterrows():
    base.loc[base['a\u00f1o'] == int(row['a\u00f1o']), 'score_entorno_u'] = row['score_entorno_u']

base['ITT'] = base.apply(_itt_row, axis=1)
base['nivel'] = base['ITT'].apply(clasificar)
print()
print('ITT actualizado con Entorno Urbano (arboles):')
print(base[['a\u00f1o','score_entorno_u','ITT','nivel']].round(1).to_string(index=False))


## Dimension Actividad Economica — Pulmon de Oriente

**Peso:** 15%  
**Indicadores:** Establecimientos activos (pos) + Empleabilidad (pos) + Ingresos log1p (pos)  
**Fuente:** Registro Mercantil 2025 — Camara de Comercio de Cali  
**Metodo:** Stock activo acumulado por año de matricula

**Refs:** Dinamicos basados en el total del stock  
- Establecimientos: [0, total_stock]  
- Empleabilidad: [0, total_personal]  
- Ingresos: [0, log1p(total_ingresos)]


In [ ]:
# ── Desarrollo Economico: Registro Mercantil ─────────────────────────
MESES_ES = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
            'julio':7,'agosto':8,'septiembre':9,'setiembre':9,
            'octubre':10,'noviembre':11,'diciembre':12}

def parse_fecha_es_de(v):
    if not v or str(v) in ('nan','None',''): return pd.NaT
    m = re.search(r'(\d{1,2})\s+de\s+([a-z\xe1\xe9\xed\xf3\xfa]+)\s+de\s+(\d{4})', str(v).lower())
    if not m: return pd.NaT
    try: return pd.Timestamp(int(m.group(3)), MESES_ES.get(m.group(2),0) or 1, int(m.group(1)))
    except: return pd.NaT

df_de = pd.DataFrame([feat['properties'] for feat in raw_rm['features']])
for col in ['INGRESOS_A','PERSONAL_O']:
    if col in df_de.columns:
        df_de[col] = pd.to_numeric(df_de[col], errors='coerce')

# Detectar campo de fecha de matricula
col_fecha_mat = None
for c in ['Mes_Dia_An','fecha_matricula','FECHA_MATRICULA']:
    if c in df_de.columns:
        col_fecha_mat = c
        break

if col_fecha_mat:
    df_de['fecha_mat'] = df_de[col_fecha_mat].apply(parse_fecha_es_de)
else:
    # Intentar parseo directo
    for c in df_de.columns:
        if 'fecha' in c.lower() or 'matri' in c.lower():
            df_de['fecha_mat'] = pd.to_datetime(df_de[c], errors='coerce')
            col_fecha_mat = c
            break

if 'fecha_mat' in df_de.columns:
    df_de['anio_mat'] = df_de['fecha_mat'].dt.year
    df_de['trim_mat'] = df_de['fecha_mat'].dt.quarter
else:
    print('AVISO: No se pudo identificar campo de fecha de matricula')
    df_de['anio_mat'] = np.nan

if 'INGRESOS_A' in df_de.columns:
    df_de['ingresos_limpio'] = df_de['INGRESOS_A'].where(df_de['INGRESOS_A'] > 0, np.nan)
else:
    df_de['ingresos_limpio'] = np.nan

if 'PERSONAL_O' in df_de.columns:
    df_de['personal_limpio'] = df_de['PERSONAL_O'].where(df_de['PERSONAL_O'] >= 0, np.nan)
else:
    df_de['personal_limpio'] = np.nan

print(f'Registro Mercantil cargado: {len(df_de)} negocios (snapshot)')
print(f'  Personal total stock: {df_de["personal_limpio"].sum():.0f}')
print(f'  Ingresos positivos: ${df_de["ingresos_limpio"].sum():,.0f}')


In [ ]:
# ── Scores Actividad Economica (stock activo acumulado) ──────────────
REF_EST_MIN = 0
REF_EST_MAX = int(len(df_de))
REF_PERS_STOCK_MIN = 0
REF_PERS_STOCK_MAX = float(df_de['personal_limpio'].sum(skipna=True)) if df_de['personal_limpio'].sum() > 0 else 1.0
REF_ING_STOCK_MIN = 0
REF_ING_STOCK_MAX = float(np.log1p(df_de['ingresos_limpio'].sum(skipna=True))) if df_de['ingresos_limpio'].sum() > 0 else 1.0

rows_de_stock = []
for anio in ANIOS:
    if anio <= 2025:
        sub_stock = df_de[df_de['anio_mat'] <= anio].copy()
        tipo = 'Real' if anio == 2025 else 'Proxy retrospectivo'
    else:
        sub_stock = df_de[df_de['anio_mat'] <= 2025].copy()
        tipo = 'Proxy referencial = 2025'

    est = float(len(sub_stock))
    pers = float(sub_stock['personal_limpio'].sum(skipna=True))
    ing = float(sub_stock['ingresos_limpio'].sum(skipna=True))
    ing_log = float(np.log1p(ing)) if ing > 0 else 0.0

    sc_neg  = score_ref(est, REF_EST_MIN, REF_EST_MAX, False)
    sc_pers = score_ref(pers, REF_PERS_STOCK_MIN, REF_PERS_STOCK_MAX, False)
    sc_ing  = score_ref(ing_log, REF_ING_STOCK_MIN, REF_ING_STOCK_MAX, False)
    vals    = [x for x in [sc_neg, sc_pers, sc_ing] if not pd.isna(x)]
    sc_de   = float(np.mean(vals)) if vals else np.nan

    rows_de_stock.append({
        'a\u00f1o': anio, 'tipo_dato': tipo,
        'establecimientos': est, 'empleabilidad': pers,
        'ingresos_log1p': ing_log,
        'score_est': sc_neg, 'score_pers': sc_pers,
        'score_ing': sc_ing, 'score_des_eco': sc_de,
    })

df_de_scores = pd.DataFrame(rows_de_stock)

for _, r in df_de_scores.iterrows():
    if not pd.isna(r['score_des_eco']):
        base.loc[base['a\u00f1o']==int(r['a\u00f1o']), 'score_des_eco'] = r['score_des_eco']

base['ITT'] = base.apply(_itt_row, axis=1)
base['nivel'] = base['ITT'].apply(clasificar)

print('Scores Actividad Economica (stock):')
print(df_de_scores[['a\u00f1o','tipo_dato','establecimientos','score_est','score_pers','score_ing','score_des_eco']].round(1).to_string(index=False))
print()
print('ITT actualizado con Act. Economica:')
cols_b = ['a\u00f1o','score_seguridad','score_movilidad','score_des_social','score_entorno_u','score_des_eco','ITT','nivel']
print(base[cols_b].round(1).to_string(index=False))


## ITT Global — Pulmon de Oriente 2026

Agregacion final de las 5 dimensiones con pesos:
- Seguridad: 27%
- Movilidad: 22%
- Desarrollo Social: 19% (Cohesion + Educacion)
- Entorno Urbano: 17%
- Actividad Economica: 15%


In [ ]:
# ── ITT Global: barras + composicion apilada ──────────────────────────
ANIOS_R   = [a for a in ANIOS if a <= 2025]
base_plot = base[base['a\u00f1o'].isin(ANIOS_R)].copy()

COLORES_ITT  = [OKI_AZUL, OKI_VERDE, OKI_NARANJA][:len(ANIOS_R)]
band_configs = [(0,40,'#FFCDD2','Activacion'),(40,60,'#FFE0B2','Consolidacion'),
                (60,80,'#C8E6C9','Transformacion'),(80,100,'#BBDEFB','Escala')]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig.suptitle('ITT Global \u2014 Pulmon de Oriente (5 dimensiones)', fontsize=13, fontweight='bold', color='#1B2631')

ax1 = axes[0]
bars = ax1.bar(ANIOS_R, base_plot['ITT'], color=COLORES_ITT, alpha=0.85, edgecolor='white', width=0.5)
for bar, val, nivel in zip(bars, base_plot['ITT'], base_plot['nivel']):
    ax1.text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.1f}\n{nivel}',
             ha='center', va='bottom', fontsize=10, fontweight='bold',
             color=NIVEL_COLORS.get(nivel,'#1B2631'))
for y0,y1,c,l in band_configs: ax1.axhspan(y0, y1, alpha=0.15, color=c)
ax1.set_title('ITT por A\u00f1o', fontweight='bold', pad=10)
ax1.set_ylim(0, 115); ax1.set_ylabel('ITT (0-100)'); ax1.set_xticks(ANIOS_R)

ax2 = axes[1]
dims  = ['score_seguridad','score_movilidad','score_des_social','score_entorno_u','score_des_eco']
lbls  = ['Seguridad','Movilidad','Des.Social','EntornoU','Act.Econ.']
pesos_l = [PESOS['Seguridad'],PESOS['Movilidad'],PESOS['DesSocial'],PESOS['EntornoU'],PESOS['DesEco']]
cols  = [OKI_BERMELL, OKI_AZUL, OKI_MORADO, OKI_VERDE, OKI_AZUL_C]

bottom = np.zeros(len(ANIOS_R))
for dim, lbl, peso, col in zip(dims, lbls, pesos_l, cols):
    if dim in base_plot.columns:
        vals = base_plot[dim].fillna(0).values * peso
        ax2.bar(ANIOS_R, vals, bottom=bottom, label=f'{lbl} ({peso:.0%})',
                color=col, alpha=0.8, edgecolor='white', width=0.5)
        bottom += vals
ax2.plot(ANIOS_R, base_plot['ITT'], 'D-', color='black', linewidth=2, markersize=8, label='ITT Total', zorder=5)
for y0,y1,c,l in band_configs: ax2.axhspan(y0, y1, alpha=0.1, color=c)
ax2.set_title('Composicion ITT (5 dimensiones)', fontweight='bold', pad=10)
ax2.set_ylim(0, 115); ax2.set_ylabel('Score ponderado'); ax2.set_xticks(ANIOS_R)
ax2.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.savefig('itt_pulmon_global.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── Evolucion Trimestral ITT Global ──────────────────────────────────
REFS_Q = {
    'homicidios':     (5/4,   50/4,  True),
    'hurtos':         (200/4, 450/4, True),
    'siniestralidad': (28/4,  75/4,  True),
    'lesionados':     (20/4,  65/4,  True),
    'mortales':       (1/4,   10/4,  True),
    'vif':            (60/4,  200/4, True),
    'rinas':          (20/4,  160/4, True),
}

rows_q = []
for _, r in corr_trim.iterrows():
    anio = int(r['a\u00f1o']); trim = int(r['trimestre'])
    if anio == 2026 and trim > 1: continue

    def qs(col):
        v = r[col]
        if pd.isna(v): return np.nan
        rmin, rmax, inv = REFS_Q[col]
        return score_ref(v, rmin, rmax, inv)

    seg_vals = [x for x in [qs('homicidios'), qs('hurtos')] if not pd.isna(x)]
    mov_vals = [x for x in [qs('siniestralidad'), qs('lesionados'), qs('mortales')] if not pd.isna(x)]
    coh_vals = [x for x in [qs('vif'), qs('rinas')] if not pd.isna(x)]

    sc_seg = np.mean(seg_vals) if seg_vals else np.nan
    sc_mov = np.mean(mov_vals) if mov_vals else np.nan

    brow   = base[base['a\u00f1o'] == anio]
    sc_spa = float(brow['score_spa'].iloc[0]) if not brow.empty and 'score_spa' in brow.columns else np.nan
    sc_coh = np.mean(coh_vals + ([sc_spa] if not pd.isna(sc_spa) else [])) if coh_vals else np.nan
    sc_ent = float(brow['score_entorno_u'].iloc[0]) if not brow.empty else np.nan
    sc_edu = float(brow['score_educ_des'].iloc[0])  if not brow.empty else np.nan
    sc_soc = np.mean([v for v in [sc_coh, sc_edu] if not pd.isna(v)]) if (not pd.isna(sc_coh) or not pd.isna(sc_edu)) else np.nan
    sc_de  = float(brow['score_des_eco'].iloc[0])   if not brow.empty else np.nan

    dims = {'Seguridad':sc_seg,'Movilidad':sc_mov,'DesSocial':sc_soc,'EntornoU':sc_ent,'DesEco':sc_de}
    total_peso = sum(PESOS[k] for k,v in dims.items() if not pd.isna(v))
    itt_q = sum(PESOS[k]*v for k,v in dims.items() if not pd.isna(v)) / total_peso if total_peso > 0 else np.nan

    rows_q.append({'periodo': r['periodo'], 'a\u00f1o': anio, 'trimestre': trim,
                   'ITT': itt_q, 'parcial': anio == 2026})

df_evol = pd.DataFrame(rows_q)
print(f'Trimestres calculados: {len(df_evol)}')
print(df_evol[['periodo','ITT']].round(1).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
ax.set_facecolor(BG)
x_idx    = list(range(len(df_evol)))
itt_vals = df_evol['ITT'].values

BANDAS = [
    (0,  40,  '#FFCDD2', 'Activacion'),
    (40, 60,  '#FFF9C4', 'Consolidacion'),
    (60, 80,  '#E8F5E9', 'Transformacion'),
    (80, 100, '#E3F2FD', 'Escala'),
]
for y0, y1, color, lbl in BANDAS:
    ax.axhspan(y0, y1, alpha=0.25, color=color, zorder=0)
    ax.text(len(df_evol)-0.3, (y0+y1)/2, lbl,
            fontsize=7.5, color='gray', ha='right', va='center', style='italic', alpha=0.85)

for y in [40, 60, 80]:
    ax.axhline(y, color='gray', linewidth=0.5, linestyle='--', alpha=0.4)

ax.plot(x_idx, itt_vals, color=OKI_AZUL, linewidth=2.5, marker='o', markersize=7, zorder=3)
ax.fill_between(x_idx, itt_vals, alpha=0.12, color=OKI_AZUL)
for xi, v in zip(x_idx, itt_vals):
    if not np.isnan(v):
        ax.text(xi, v+1.5, f'{v:.1f}', ha='center', fontsize=7.5, fontweight='bold', color=OKI_AZUL)

# Separadores por a\u00f1o
for yr in df_evol['a\u00f1o'].unique():
    mask = df_evol['a\u00f1o'] == yr
    xi_start = df_evol[mask].index[0]
    ax.axvline(xi_start-0.5, color='#CCCCCC', linewidth=0.8, linestyle='-', alpha=0.5)
    ax.text(xi_start+0.2, 102, str(yr), fontsize=9, fontweight='bold', color='#555555')

ax.set_xticks(x_idx)
ax.set_xticklabels(df_evol['periodo'], rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 108)
ax.set_ylabel('ITT (0-100)')
ax.set_title('Evolucion Trimestral ITT \u2014 Pulmon de Oriente (5 dimensiones)',
             fontsize=13, fontweight='bold', color='#1B2631', pad=12)
plt.tight_layout()
plt.savefig('itt_pulmon_evol_trim.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── Cards de metricas clave ───────────────────────────────────────────
ANIOS_3 = [a for a in ANIOS if a <= 2025]
if len(ANIOS_3) >= 3:
    d_ini = base[base['a\u00f1o']==ANIOS_3[0]].iloc[0]
    d_ant = base[base['a\u00f1o']==ANIOS_3[-2]].iloc[0]
    d_ult = base[base['a\u00f1o']==ANIOS_3[-1]].iloc[0]

    cards = [
        ('Homicidios',    int(d_ini['homicidios']),   int(d_ant['homicidios']),   int(d_ult['homicidios']),    True),
        ('Hurtos',        int(d_ini['hurtos']),        int(d_ant['hurtos']),        int(d_ult['hurtos']),         True),
        ('Siniestros',    int(d_ini['siniestralidad']),int(d_ant['siniestralidad']),int(d_ult['siniestralidad']), True),
        ('VIF',           int(d_ini['vif']),           int(d_ant['vif']),           int(d_ult['vif']),            True),
        ('Rinas',         int(d_ini['rinas']),         int(d_ant['rinas']),         int(d_ult['rinas']),          True),
        ('Sc.Seguridad',  d_ini['score_seguridad'],   d_ant['score_seguridad'],   d_ult['score_seguridad'],     False),
        ('Sc.Movilidad',  d_ini['score_movilidad'],   d_ant['score_movilidad'],   d_ult['score_movilidad'],     False),
        ('ITT Global',    d_ini['ITT'],               d_ant['ITT'],               d_ult['ITT'],                 False),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(18, 7), facecolor=BG)
    fig.suptitle(f'ITT Pulmon de Oriente \u2014 Metricas Clave | {ANIOS_3[0]}-{ANIOS_3[-1]}',
                 fontsize=14, fontweight='bold', color='#1B2631', y=0.98)
    for i, (titulo, v_ini, v_ant, v_ult, inv) in enumerate(cards):
        ax = axes[i//4][i%4]; ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
        rect = mpatches.FancyBboxPatch((0.02,0.02),0.96,0.96, boxstyle='round,pad=0.02',
                linewidth=1.5, edgecolor='#DEE2E6', facecolor='white')
        ax.add_patch(rect)
        ax.text(0.5,0.85, titulo, ha='center', fontsize=9, color='#6C757D', fontweight='bold')
        val_d = f'{v_ult:.1f}' if isinstance(v_ult,float) else str(v_ult)
        ax.text(0.5,0.60, val_d, ha='center', fontsize=19, fontweight='bold', color='#1B2631')
        pct1 = safe_pct(v_ult, v_ant)
        ar1, col1 = arrow(pct1, inv)
        ax.text(0.5,0.38, f'{ANIOS_3[-1]} vs {ANIOS_3[-2]}: {ar1}', ha='center', fontsize=8, color=col1, fontweight='bold')
        pct2 = safe_pct(v_ult, v_ini)
        ar2, col2 = arrow(pct2, inv)
        ax.text(0.5,0.22, f'vs {ANIOS_3[0]}: {ar2}', ha='center', fontsize=7.5, color=col2)
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig('itt_pulmon_cards.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
else:
    print('Se requieren al menos 3 a\u00f1os para cards comparativas')


## Exportacion Excel — 9 hojas

Genera un archivo Excel consolidado con las siguientes hojas:
1. Resumen (ITT global por año)
2. Trimestral (evolucion trimestral)
3. Seguridad (detalle indicadores)
4. Movilidad (detalle indicadores)
5. Cohesion_Social (VIF + Rinas + SPA)
6. Entorno_Urbano (NDVI + Arboles)
7. Educacion (Desercion + Repitencia)
8. Desarrollo_Economico (stock activo)
9. Indicadores_Trimestrales (serie cruda)


In [ ]:
EXPORT_PATH = 'ITT_Pulmon_de_Oriente_2026.xlsx'

with pd.ExcelWriter(EXPORT_PATH, engine='openpyxl') as writer:
    # Hoja 1: Resumen
    cols_resumen = ['a\u00f1o','score_seguridad','score_movilidad','score_des_social',
                    'score_entorno_u','score_des_eco','ITT','nivel']
    base[cols_resumen].round(2).to_excel(writer, sheet_name='Resumen', index=False)

    # Hoja 2: Trimestral
    if 'df_evol' in dir() and len(df_evol) > 0:
        df_evol.round(2).to_excel(writer, sheet_name='Trimestral', index=False)

    # Hoja 3: Seguridad
    seg_cols = ['a\u00f1o','homicidios','hurtos','score_homicidios','score_hurtos','score_seguridad']
    base[[c for c in seg_cols if c in base.columns]].round(2).to_excel(writer, sheet_name='Seguridad', index=False)

    # Hoja 4: Movilidad
    mov_cols = ['a\u00f1o','siniestralidad','lesionados','mortales','score_siniestralidad','score_lesionados','score_mortales','score_movilidad']
    base[[c for c in mov_cols if c in base.columns]].round(2).to_excel(writer, sheet_name='Movilidad', index=False)

    # Hoja 5: Cohesion Social
    coh_cols = ['a\u00f1o','vif','rinas','spa','score_vif','score_rinas','score_spa','score_cohesion']
    base[[c for c in coh_cols if c in base.columns]].round(2).to_excel(writer, sheet_name='Cohesion_Social', index=False)

    # Hoja 6: Entorno Urbano
    if 'df_eu' in dir():
        df_eu.round(2).to_excel(writer, sheet_name='Entorno_Urbano', index=False)

    # Hoja 7: Educacion
    edu_data = pd.DataFrame({'a\u00f1o': ANIOS, 'score_educ_des': REF_EDUC_DES, 'fuente': 'Referente provisional'})
    edu_data.to_excel(writer, sheet_name='Educacion', index=False)

    # Hoja 8: Desarrollo Economico
    if 'df_de_scores' in dir():
        df_de_scores.round(2).to_excel(writer, sheet_name='Desarrollo_Economico', index=False)

    # Hoja 9: Indicadores Trimestrales
    corr_trim.round(2).to_excel(writer, sheet_name='Indicadores_Trimestrales', index=False)

print(f'\u2705 Excel exportado: {EXPORT_PATH}')
print(f'   Hojas: Resumen, Trimestral, Seguridad, Movilidad, Cohesion_Social,')
print(f'          Entorno_Urbano, Educacion, Desarrollo_Economico, Indicadores_Trimestrales')


In [ ]:
# ── Mapa interactivo del poligono ─────────────────────────────────────
import folium

centroide = gdf_pulmon_wgs.geometry.centroid.iloc[0]
m = folium.Map(location=[centroide.y, centroide.x], zoom_start=13, tiles='CartoDB positron')

folium.GeoJson(
    gdf_pulmon_wgs.__geo_interface__,
    name='Pulmon de Oriente',
    style_function=lambda x: {'fillColor': OKI_AZUL, 'color': OKI_AZUL,
                              'weight': 2, 'fillOpacity': 0.15}
).add_to(m)

# Puntos de homicidios
gdf_hom = gpd.GeoDataFrame.from_features(raw_hom['features'], crs='EPSG:4326')
for _, row in gdf_hom.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color=OKI_BERMELL, fill=True, fill_opacity=0.7,
        popup='Homicidio'
    ).add_to(m)

folium.LayerControl().add_to(m)
print(f'Mapa generado: centroide ({centroide.y:.4f}, {centroide.x:.4f})')
m


## Notas Metodologicas

### Datos disponibles vs provisionales

| Dimension | Estado | Fuente |
|---|---|---|
| Seguridad | **Dato real** | DATIC 2023-2026 T1 |
| Movilidad | **Dato real** | BD Siniestros 2023-2026 |
| Cohesion Social | **Dato real parcial** | VIF real + Rinas real + SPA real |
| Entorno Urbano | **Parcial** | Arboles DAGMA real. NDVI pendiente TIF |
| Educacion | Referente provisional | 54.9 (Pulmon de Oriente T4-2025) |
| Act. Economica | **Dato real** | Registro Mercantil 2025 |

### Reglas aplicadas
- `ref_min/ref_max` fijos calibrados para zona grande (>100K hab)
- No se usa min-max relativo
- 2026: solo Q1 disponible para DATIC. Q2-Q4 = NaN
- Paleta Okabe-Ito para accesibilidad visual
- Heatmaps con cmap cividis (perceptualmente uniforme)
- Indicadores anuales (Entorno, Educacion, Act.Economica) extendidos planos a 4 trimestres

### Pendientes
- [ ] Incluir rasters NDVI Sentinel-2 para Pulmon de Oriente (2023-2026)
- [ ] Identificar sedes educativas en el poligono para calculo SIMAT/SIMPADE real
- [ ] Validar campo fecha del Registro Mercantil para serie temporal correcta
- [ ] Incorporar Deficit Habitacional AHDI si aplica
